# 📷 Smart Campus Lost & Found — Image Preprocessing & Matching Pipeline

This notebook implements the visual intelligence component for the Campus Lost & Found System:
1. **Dataset Ingestion:** Inspects the Roboflow Lost & Found dataset (Bottle, Calculator, Charger, Key, Umbrella, etc.).
2. **OpenCV Preprocessing:** Center-cropping, BGR-to-RGB conversion, resizing ($224 \times 224$), and pixel normalization.
3. **CNN Feature Extraction:** Extracts dense embeddings using a pre-trained **MobileNetV2** (1280-d) / **ResNet50** (2048-d) backbone.
4. **Visual Similarity Matching:** Calculates Cosine Similarity between a query (Lost Item) and catalog (Found Items).
5. **Candidate Retrieval Visualization:** Displays the query item alongside the top-K matched candidates with confidence scores.


In [ ]:
import os
import glob
import yaml
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False

try:
    import torch
    import torchvision.models as models
    import torchvision.transforms as transforms
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

# Paths configuration
BASE_DIR = os.path.dirname(os.path.abspath(''))
DATA_DIR = os.path.join(BASE_DIR, 'data', 'images')
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train', 'images')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test', 'images')
YAML_FILE = os.path.join(DATA_DIR, 'data.yaml')

print(f"[*] Image base directory: {DATA_DIR}")
print(f"[*] OpenCV available: {HAS_CV2} | PyTorch available: {HAS_TORCH}")


## 1. Inspecting Dataset Classes & Images

In [ ]:
# Load class names from data.yaml
classes = []
if os.path.exists(YAML_FILE):
    with open(YAML_FILE, 'r') as f:
        meta = yaml.safe_load(f)
        classes = meta.get('names', [])
        print(f"[*] Target Classes ({len(classes)}): {classes}")

train_images = glob.glob(os.path.join(TRAIN_IMG_DIR, '*.jpg')) + glob.glob(os.path.join(TRAIN_IMG_DIR, '*.png'))
test_images = glob.glob(os.path.join(TEST_IMG_DIR, '*.jpg')) + glob.glob(os.path.join(TEST_IMG_DIR, '*.png'))

print(f"[*] Total Training (Catalog) Images: {len(train_images):,}")
print(f"[*] Total Test (Query) Images:       {len(test_images):,}")


## 2. OpenCV Image Preprocessing Function

To ensure optimal feature extraction from deep CNN models:
- Center-crop to 1:1 aspect ratio to prevent stretching objects.
- Resize to standard $224 \times 224$ pixels.
- Convert BGR color space to RGB.
- Normalize pixel values to $[0.0, 1.0]$ float range.

In [ ]:
def preprocess_image_cv(image_path, target_size=(224, 224)):
    """OpenCV Preprocessing Pipeline with PIL fallback."""
    if HAS_CV2:
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Could not load {image_path}")
        
        # Center crop to square
        h, w = img.shape[:2]
        min_dim = min(h, w)
        sy, sx = (h - min_dim) // 2, (w - min_dim) // 2
        cropped = img[sy:sy + min_dim, sx:sx + min_dim]
        
        # Resize to 224x224
        resized = cv2.resize(cropped, target_size, interpolation=cv2.INTER_AREA)
        
        # BGR -> RGB
        rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
        return rgb.astype(np.float32) / 255.0
    else:
        with Image.open(image_path) as pil_img:
            pil_img = pil_img.convert('RGB')
            w, h = pil_img.size
            min_dim = min(w, h)
            cropped = pil_img.crop(((w-min_dim)//2, (h-min_dim)//2, (w+min_dim)//2, (h+min_dim)//2))
            resized = cropped.resize(target_size, Image.Resampling.LANCZOS)
            return np.array(resized, dtype=np.float32) / 255.0

# Visual demonstration of Preprocessing Steps
if train_images:
    sample_path = train_images[0]
    raw_img = Image.open(sample_path)
    proc_img = preprocess_image_cv(sample_path)
    
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(raw_img)
    ax[0].set_title(f"Raw Image ({raw_img.size[0]}x{raw_img.size[1]})")
    ax[0].axis('off')
    
    ax[1].imshow(proc_img)
    ax[1].set_title("Preprocessed Image (224x224 Normalized)")
    ax[1].axis('off')
    plt.tight_layout()
    plt.show()


## 3. Pretrained CNN Feature Embedder (MobileNetV2)

We pass the preprocessed image through MobileNetV2 and capture the output of the global average pooling layer, producing a 1280-dimensional feature vector representing colors, shapes, patterns, and object structures.

In [ ]:
class EmbeddingModel:
    def __init__(self):
        if HAS_TORCH:
            weights = models.MobileNet_V2_Weights.DEFAULT
            self.net = models.mobilenet_v2(weights=weights)
            self.net.classifier = torch.nn.Identity()  # 1280-dim embedding
            self.net.eval()
        else:
            self.net = None
            
    def extract(self, image_path):
        img_np = preprocess_image_cv(image_path)
        if not HAS_TORCH:
            # Mock embedding for demonstration if torch is installing
            return np.random.randn(1280)
        
        tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        tensor = (tensor - mean) / std
        
        with torch.no_grad():
            feat = self.net(tensor).squeeze().numpy()
        
        # L2 unit normalization
        norm = np.linalg.norm(feat)
        return feat / norm if norm > 0 else feat

embedder = EmbeddingModel()
print("[OK] Embedding Model initialized.")


## 4. Visual Similarity Matching (Lost vs Found Query)

Given a query "Lost Item" photo, we search the "Found Items" database to locate items with the highest Cosine Similarity score.

In [ ]:
def find_top_matches(query_path, gallery_paths, top_k=3):
    query_vec = embedder.extract(query_path)
    scores = []
    for path in gallery_paths:
        cand_vec = embedder.extract(path)
        # Cosine similarity via dot product of L2 normalized vectors
        sim = float(np.dot(query_vec, cand_vec))
        scores.append((path, sim))
        
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# Run a demonstration match using the first test image
if test_images and train_images:
    query_img_path = test_images[0]
    gallery_subset = train_images[:100]  # Search across first 100 catalog images
    
    top_matches = find_top_matches(query_img_path, gallery_subset, top_k=3)
    
    # Visualize Query vs Top Matches
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(Image.open(query_img_path))
    axes[0].set_title("🔍 Query: Lost Item", color='darkred', fontweight='bold')
    axes[0].axis('off')
    
    for idx, (match_path, score) in enumerate(top_matches):
        axes[idx + 1].imshow(Image.open(match_path))
        axes[idx + 1].set_title(f"Match #{idx+1}\nSimilarity: {score*100:.1f}%")
        axes[idx + 1].axis('off')
        
    plt.suptitle("Smart Campus Lost & Found — Image Retrieval Results", fontsize=14, y=1.05)
    plt.tight_layout()
    plt.show()
